# NhuLe pipeline

In [1]:
!git clone https://github.com/nhatvu205/vi-multimodal-sacarsm-detection-on-social-media.git
%cd vi-multimodal-sacarsm-detection-on-social-media

Cloning into 'vi-multimodal-sacarsm-detection-on-social-media'...
remote: Enumerating objects: 1276, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 1276 (delta 131), reused 113 (delta 109), pack-reused 1095 (from 1)
Receiving objects: 100% (1276/1276), 31.81 MiB | 29.77 MiB/s, done.
Resolving deltas: 100% (715/715), done.
/kaggle/working/vi-multimodal-sacarsm-detection-on-social-media


In [2]:
!git checkout nhule

Branch 'nhule' set up to track remote branch 'nhule' from 'origin'.
Switched to a new branch 'nhule'


In [3]:
!git branch

  main
* nhule


In [4]:
!ls

data		    label-studio-setup	round-2-annotation
emoji_processor.py  preprocessing	sarcasm_experiment.py
experiment_setup    requirements.txt
facebook_scraper    round-1-annotation


In [5]:
# =============================================================================
# CELL 1 — Cài đặt các thư viện cần thiết
# =============================================================================
!pip install emoji scikit-learn transformers -q

In [6]:
# =============================================================================
# CELL 2 — Convert JSON → CSV (format mà sarcasm_experiment.py cần)
# =============================================================================
import json
import csv
import os
from pathlib import Path

DATASET_ROOT = "/kaggle/input/datasets/nhatvu205/sacasm-dataset-uit" 
IMAGE_DIR    = f"{DATASET_ROOT}/images"
DATA_DIR     = "data"
os.makedirs(DATA_DIR, exist_ok=True)

SPLIT_FILES = {
    "train": f"{DATASET_ROOT}/final-data/train.json",
    "val":   f"{DATASET_ROOT}/final-data/dev.json",
    "test":  f"{DATASET_ROOT}/final-data/test.json",
}

def convert_json_to_csv(json_path: str, csv_path: str, label_field: str = "mm_label"):
    """
    Convert từ JSON của dataset gốc sang CSV format:
      text, image_path, label
    image_path chỉ giữ tên file (không có prefix data/images/)
    vì SarcasmDataset sẽ join với CFG['image_dir'].
    """
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["text", "image_path", "label"])
        writer.writeheader()
        for sample in data:
            # Lấy tên file ảnh, bỏ prefix thừa
            img_path = sample.get("image_path", "")
            img_name = Path(img_path).name   # vd: post00001.jpg
            writer.writerow({
                "text":       sample.get("text", ""),
                "image_path": img_name,
                "label":      sample.get(label_field, 0),
            })

    print(f"  ✔ {json_path} → {csv_path}  ({len(data)} samples)")

print("Converting JSON → CSV...")
for split, json_path in SPLIT_FILES.items():
    convert_json_to_csv(json_path, f"{DATA_DIR}/{split}.csv", label_field="mm_label")

# Kiểm tra nhanh
import pandas as pd
df = pd.read_csv("data/train.csv")
print(f"\nTrain: {len(df)} rows | Label dist: {df['label'].value_counts().to_dict()}")
df.head(3)

Converting JSON → CSV...
  ✔ /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/final-data/train.json → data/train.csv  (5884 samples)
  ✔ /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/final-data/dev.json → data/val.csv  (735 samples)
  ✔ /kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/final-data/test.json → data/test.csv  (736 samples)

Train: 5884 rows | Label dist: {0: 3305, 1: 2579}


,text,image_path,label
0,"Thứ tôi theo đuổi không phải là ai đó,\nmà là ...",post03261.jpg,0
1,Nuôi tôi luôn được không ạ,post06616.jpg,1
2,"mình là Nam, mình vừa cầu hôn bạn gái mình thà...",post04262.jpg,0


In [7]:
# =============================================================================
# CELL 3 — Kiểm tra ảnh có tồn tại không
# =============================================================================
df_test = pd.read_csv("data/test.csv")
missing = [r for r in df_test["image_path"] if not Path(f"{IMAGE_DIR}/{r}").exists()]
print(f"Missing images: {len(missing)}/{len(df_test)}")
if missing:
    print("Sample missing:", missing[:5])

Missing images: 0/736


In [8]:
# =============================================================================
# CELL 3.5 — Patch sarcasm_experiment.py: stub LLaVA & QwenVL chưa implement
# =============================================================================
import re

with open("sarcasm_experiment.py", "r", encoding="utf-8") as f:
    code = f.read()

# Thêm stub class trước MULTI_MODELS dict
stub = '''
class LLaVAClassifier:
    """Stub — chưa implement, bỏ qua khi chạy text-only."""
    pass

class QwenVLClassifier:
    """Stub — chưa implement, bỏ qua khi chạy text-only."""
    pass

'''

# Chèn stub trước dòng MULTI_MODELS
code = code.replace(
    'MULTI_MODELS = {',
    stub + 'MULTI_MODELS = {'
)

with open("sarcasm_experiment.py", "w", encoding="utf-8") as f:
    f.write(code)

print("✅ Đã patch LLaVAClassifier + QwenVLClassifier stub")

✅ Đã patch LLaVAClassifier + QwenVLClassifier stub


In [9]:
# =============================================================================
# CELL 4 — Patch CFG: trỏ image_dir đúng vào Kaggle input
# =============================================================================
# Chạy cell này TRƯỚC khi import sarcasm_experiment
import sys
sys.path.insert(0, ".")   # đảm bảo import emoji_processor được

In [10]:
# Patch CFG trước khi module load
import importlib
import sarcasm_experiment as exp

exp.CFG["data_dir"]    = "data"
exp.CFG["image_dir"]   = IMAGE_DIR
exp.CFG["batch_size"]  = 16
exp.CFG["epochs"]      = 5
exp.CFG["lr"]          = 2e-5
exp.CFG["num_workers"] = 2

print("CFG:", exp.CFG)
print("Device:", exp.CFG["device"])

2026-05-30 21:55:48,762 [WARNING] ViSoLex không tìm thấy. Cài bằng: pip install visolex
  → Scenario s2/s4 sẽ bỏ qua bước chuẩn hoá từ vựng.


CFG: {'data_dir': 'data', 'image_dir': '/kaggle/input/datasets/nhatvu205/sacasm-dataset-uit/images', 'checkpoint_dir': 'checkpoints', 'output_dir': 'results', 'batch_size': 16, 'max_len': 128, 'epochs': 5, 'lr': 2e-05, 'warmup_ratio': 0.1, 'seed': 42, 'device': 'cuda', 'image_size': 224, 'num_workers': 2}
Device: cuda


In [14]:
# =============================================================================
# CELL 5 — Chạy PhoBERT 
# =============================================================================
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
 
print("\n" + "="*60)
print("  RUNNING: PhoBERT-base")
print("="*60)
 
phobert_results = exp.run_text_model(
    model_key = "phobert-base",
    scenarios = ["s1", "s2", "s3", "s4"],
)
 
print("\n✅ PhoBERT done:")
for sc, m in phobert_results.items():
    print(f"  {sc}: F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}  AUC={m.get('auc', 0):.4f}")

2026-05-30 18:16:06,053 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-30 18:16:06,055 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-30 18:16:06,069 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/phobert-base/c1e37c5c86f918761049cef6fa216b4779d0d01d/config.json "HTTP/1.1 200 OK"


GPU: Tesla T4

  RUNNING: PhoBERT-base


2026-05-30 18:16:06,083 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/vinai/phobert-base/c1e37c5c86f918761049cef6fa216b4779d0d01d/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

2026-05-30 18:16:06,188 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-30 18:16:06,279 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-30 18:16:06,365 [INFO] HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-30 18:16:06,450 [INFO] HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-05-30 18:16:06,528 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base/resolve/main/vocab.txt "HTTP/1.1 307 Temporary Redirect"
2026-05-30 18:16:06,542 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/phobert-base/c1e37c5c86f918761049cef6fa216b4779d0d01d/vocab.txt "HTTP/1.1 200 OK"
2026-05-30 

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-30 18:41:29,296 [INFO] HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base/commits/refs%2Fpr%2F7 "HTTP/1.1 200 OK"
RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.weight          | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-30 18:41:29,456 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base/resolve/refs%2Fpr%2F7/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-05-3

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-30 18:55:09,717 [INFO] HTTP Request: GET https://huggingface.co/api/models/vinai/phobert-base/commits/refs%2Fpr%2F7 "HTTP/1.1 200 OK"
RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.weight          | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-30 18:55:09,851 [INFO] HTTP Request: HEAD https://huggingface.co/vinai/phobert-base/resolve/refs%2Fpr%2F7/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-05-3


✅ PhoBERT done:
  s1: F1=0.6873  Acc=0.6861  AUC=0.7552
  s2: F1=0.6882  Acc=0.6889  AUC=0.7519
  s3: F1=0.6747  Acc=0.6753  AUC=0.7407
  s4: F1=0.6814  Acc=0.6807  AUC=0.7461


In [15]:
# =============================================================================
# CELL 6 — Chạy mBERT
# =============================================================================
print("\n" + "="*60)
print("  RUNNING: mBERT")
print("="*60)

mbert_results = exp.run_text_model(
    model_key = "mbert",
    scenarios = ["s1", "s2", "s3", "s4"],
)

print("\n✅ mBERT done:")
for sc, m in mbert_results.items():
    print(f"  {sc}: F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}  AUC={m.get('auc', 0):.4f}")


2026-05-30 19:08:46,855 [INFO] HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"



  RUNNING: mBERT


2026-05-30 19:08:46,943 [INFO] HTTP Request: GET https://huggingface.co/bert-base-multilingual-cased/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

2026-05-30 19:08:47,044 [INFO] HTTP Request: HEAD https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-30 19:08:47,135 [INFO] HTTP Request: GET https://huggingface.co/bert-base-multilingual-cased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-30 19:38:06,272 [INFO]   [mbert|s3] Epoch 1/5 loss=0.6427 val_f1=0.7008
2026-05-30 19:38:09,755 [INFO]   ✔ Checkpoint saved → checkpoints/mbert_s3.pt
2026-05-30 19:40:51,275 [INFO]   [mbert|s3] Epoch 2/5 loss=0.5851 val_f1=0.7075
2026-05-30 19:40:55,959 [I

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-05-30 19:52:05,661 [INFO]   [mbert|s4] Epoch 1/5 loss=0.6504 val_f1=0.5777
2026-05-30 19:52:08,717 [INFO]   ✔ Checkpoint saved → checkpoints/mbert_s4.pt
2026-05-30 19:54:51,623 [INFO]   [mbert|s4] Epoch 2/5 loss=0.5865 val_f1=0.7028
2026-05-30 19:54:55,886 [I


✅ mBERT done:
  s1: F1=0.6792  Acc=0.6780  AUC=0.7342
  s2: F1=0.6697  Acc=0.6698  AUC=0.7251
  s3: F1=0.6823  Acc=0.6834  AUC=0.7334
  s4: F1=0.6755  Acc=0.6753  AUC=0.7295


In [11]:
# =============================================================================
# CELL 7 — Chạy RoBERTa-base
# =============================================================================
print("\n" + "="*60)
print("  RUNNING: RoBERTa-base")
print("="*60)

roberta_results = exp.run_text_model(
    model_key = "roberta-base",
    scenarios = ["s1", "s2", "s3", "s4"],
)

print("\n✅ RoBERTa done:")
for sc, m in roberta_results.items():
    print(f"  {sc}: F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}  AUC={m.get('auc', 0):.4f}")

2026-05-30 21:56:03,489 [INFO] HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
2026-05-30 21:56:03,490 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-30 21:56:03,514 [INFO] HTTP Request: GET https://huggingface.co/roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"



  RUNNING: RoBERTa-base


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

2026-05-30 21:56:03,551 [INFO] HTTP Request: HEAD https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-30 21:56:03,574 [INFO] HTTP Request: GET https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-05-30 22:23:34,535 [INFO]   [roberta-base|s3] Epoch 1/5 loss=0.6585 val_f1=0.6874
2026-05-30 22:23:36,420 [INFO]   ✔ Checkpoint saved → checkpoints/roberta-base_s3.pt
2026-05-30 22:26:10,496 [INFO]   [roberta-base|s3] Epoch

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-05-30 22:36:57,610 [INFO]   [roberta-base|s4] Epoch 1/5 loss=0.6635 val_f1=0.6224
2026-05-30 22:36:59,507 [INFO]   ✔ Checkpoint saved → checkpoints/roberta-base_s4.pt
2026-05-30 22:39:38,338 [INFO]   [roberta-base|s4] Epoch


✅ RoBERTa done:
  s1: F1=0.6693  Acc=0.6685  AUC=0.7423
  s2: F1=0.6933  Acc=0.6929  AUC=0.7352
  s3: F1=0.6945  Acc=0.6943  AUC=0.7336
  s4: F1=0.6772  Acc=0.6780  AUC=0.7138


In [13]:
# =============================================================================
# CELL 8 — Tổng hợp kết quả & in bảng so sánh
# =============================================================================
all_results = {
    "text": {
#        "phobert-base": phobert_results,
#        "mbert":        mbert_results,
        "roberta-base": roberta_results,
    }
}

exp.save_results(all_results)

# In bảng đẹp
print("\n" + "="*75)
print(f"{'Model':<15} {'Scenario':<10} {'F1':>8} {'Accuracy':>10} {'AUC':>8}")
print("-"*75)
for model_name, scenarios in all_results["text"].items():
    for sc, m in scenarios.items():
        auc = f"{m['auc']:.4f}" if m.get("auc") else "  N/A"
        print(f"{model_name:<15} {sc:<10} {m['f1']:>8.4f} {m['accuracy']:>10.4f} {auc:>8}")
    print()
print("="*75)

2026-05-30 22:48:28,323 [INFO] 
📊 All results saved → results/results_20260530_224828.json



Model                Scenario       F1      Acc     Prec      Rec      AUC
------------------------------------------------------------------------------------------
roberta-base         s1         0.6693   0.6685   0.6709   0.6685   0.7423
roberta-base         s2         0.6933   0.6929   0.6939   0.6929   0.7352
roberta-base         s3         0.6945   0.6943   0.6949   0.6943   0.7336
roberta-base         s4         0.6772   0.6780   0.6768   0.6780   0.7138

Model           Scenario         F1   Accuracy      AUC
---------------------------------------------------------------------------
roberta-base    s1           0.6693     0.6685   0.7423
roberta-base    s2           0.6933     0.6929   0.7352
roberta-base    s3           0.6945     0.6943   0.7336
roberta-base    s4           0.6772     0.6780   0.7138



In [ ]:
# =============================================================================
# CELL 9 — Lưu kết quả & download
# =============================================================================
import shutil, os
from IPython.display import FileLink

# Zip checkpoints + results
shutil.make_archive("text_only_results", "zip", ".", "results")
shutil.make_archive("text_only_checkpoints", "zip", ".", "checkpoints")

print("📦 Files sẵn sàng download:")
display(FileLink("text_only_results.zip"))
display(FileLink("text_only_checkpoints.zip"))

# NhatVu pipeline

In [1]:
!git clone https://github.com/nhatvu205/vi-multimodal-sacarsm-detection-on-social-media.git
%cd vi-multimodal-sacarsm-detection-on-social-media

Cloning into 'vi-multimodal-sacarsm-detection-on-social-media'...
remote: Enumerating objects: 1471, done.
remote: Counting objects: 100% (383/383), done.
remote: Compressing objects: 100% (206/206), done.
remote: Total 1471 (delta 246), reused 262 (delta 175), pack-reused 1088 (from 1)
Receiving objects: 100% (1471/1471), 33.05 MiB | 24.21 MiB/s, done.
Resolving deltas: 100% (827/827), done.
/kaggle/working/vi-multimodal-sacarsm-detection-on-social-media


In [2]:
!git checkout nvu

error: pathspec 'nvu' did not match any file(s) known to git


In [3]:
%cd experiment_setup

/kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/experiment_setup


In [4]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 440.7/440.7 kB 8.6 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 69.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 3.3 MB/s eta 0:00:00
  Created wheel for visonorm: filename=visonorm-0.1.4-py3-none-any.whl size=472044 sha256=a6309766b941a345562b7ad54cbc8d98d0fad83abe80179bb692c9f2be108887
  Stored in directory: /root/.cache/pip/wheels/85/76/49/9ec06582b8ddbdab622058a71e995cbb96d8d9ce4d95382d4a
Successfully built visonorm
  Attempting uninstall: PyYAML
    Found existing installation: P

In [5]:
%cd ..

/kaggle/working/vi-multimodal-sacarsm-detection-on-social-media


In [10]:
!python -m experiment_setup.main \
  --config experiment_setup/configs/base.yaml \
  --stage preprocess

[run] output dir: /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/experiment_setup/runs/base_experiment
[cache] preparing train split...
2026-06-02 01:56:55,956 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-02 01:56:55,959 - datasets - INFO - JAX version 0.7.2 available.
2026-06-02 01:56:59,666 - visonorm.normalizer - INFO - Loading tokenizer and model from visolex/visobert-normalizer-mix100...
2026-06-02 01:56:59,853 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/visolex/visobert-normalizer-mix100/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-02 01:56:59,853 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-02 01:56:59,869 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/visolex/visobert-normalizer-mix100/86731c2aa82948bd3bcbb682759d62c9c2475886/co

In [8]:
!python -m experiment_setup.main \
  --config experiment_setup/configs/models/phobert_base.yaml \
  --stage all

[run] output dir: /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/experiment_setup/runs/phobert_base_mm
[cache] preparing train split...
2026-05-31 09:04:45,186 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-05-31 09:04:45,187 - datasets - INFO - JAX version 0.7.2 available.
2026-05-31 09:04:46,239 - visonorm.normalizer - INFO - Loading tokenizer and model from visolex/visobert-normalizer-mix100...
2026-05-31 09:04:46,416 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/visolex/visobert-normalizer-mix100/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-31 09:04:46,432 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/visolex/visobert-normalizer-mix100/86731c2aa82948bd3bcbb682759d62c9c2475886/config.json "HTTP/1.1 200 OK"
2026-05-31 09:04:46,494 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/visolex/visobert-normalizer-mix100/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary

In [9]:
!python -m experiment_setup.main \
  --config experiment_setup/configs/models/mbert.yaml \
  --stage all

[run] output dir: /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/experiment_setup/runs/mbert_mm
[cache] preparing train split...
2026-05-31 09:43:22,444 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-05-31 09:43:22,445 - datasets - INFO - JAX version 0.7.2 available.
2026-05-31 09:43:23,531 - visonorm.normalizer - INFO - Loading tokenizer and model from visolex/visobert-normalizer-mix100...
2026-05-31 09:43:23,703 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/visolex/visobert-normalizer-mix100/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-31 09:43:23,703 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-31 09:43:23,719 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/visolex/visobert-normalizer-mix100/86731c2aa82948bd3bcbb682759d62c9c2475886/config.js

In [10]:
!python -m experiment_setup.main \
  --config experiment_setup/configs/models/roberta_base.yaml \
  --stage all

[run] output dir: /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/experiment_setup/runs/roberta_base_mm
[cache] preparing train split...
2026-05-31 10:31:38,593 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-05-31 10:31:38,596 - datasets - INFO - JAX version 0.7.2 available.
2026-05-31 10:31:39,763 - visonorm.normalizer - INFO - Loading tokenizer and model from visolex/visobert-normalizer-mix100...
2026-05-31 10:31:39,930 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/visolex/visobert-normalizer-mix100/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-31 10:31:39,947 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/visolex/visobert-normalizer-mix100/86731c2aa82948bd3bcbb682759d62c9c2475886/config.json "HTTP/1.1 200 OK"
2026-05-31 10:31:40,011 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/visolex/visobert-normalizer-mix100/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary

In [16]:
# In file chứa các chỉ số độ chính xác (Accuracy, F1-score, Loss,...)
!cat /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/experiment_setup/runs/mbert_mm/mbert/summary.json

[
  {
    "model": "mbert",
    "scenario": "s1",
    "split": "dev",
    "accuracy": 0.6993,
    "f1_macro": 0.6978,
    "f1_weighted": 0.7005,
    "precision_weighted": 0.7055,
    "recall_weighted": 0.6993,
    "confusion_matrix": [
      [
        283,
        130
      ],
      [
        91,
        231
      ]
    ],
    "num_samples": 735,
    "auc": 0.7394
  },
  {
    "model": "mbert",
    "scenario": "s1",
    "split": "test",
    "accuracy": 0.6671,
    "f1_macro": 0.665,
    "f1_weighted": 0.6683,
    "precision_weighted": 0.6722,
    "recall_weighted": 0.6671,
    "confusion_matrix": [
      [
        275,
        139
      ],
      [
        106,
        216
      ]
    ],
    "num_samples": 736,
    "auc": 0.7168
  },
  {
    "model": "mbert",
    "scenario": "s2",
    "split": "dev",
    "accuracy": 0.6925,
    "f1_macro": 0.6919,
    "f1_weighted": 0.6936,
    "precision_weighted": 0.7033,
    "recall_weighted": 0.6925,
    "confusion_matrix": [
      [
        271,
  

In [15]:
!cat /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/experiment_setup/runs/phobert_base_mm/phobert-base/summary.json

[
  {
    "model": "phobert-base",
    "scenario": "s1",
    "split": "dev",
    "accuracy": 0.7007,
    "f1_macro": 0.6987,
    "f1_weighted": 0.7017,
    "precision_weighted": 0.7054,
    "recall_weighted": 0.7007,
    "confusion_matrix": [
      [
        287,
        126
      ],
      [
        94,
        228
      ]
    ],
    "num_samples": 735,
    "auc": 0.7661
  },
  {
    "model": "phobert-base",
    "scenario": "s1",
    "split": "test",
    "accuracy": 0.6698,
    "f1_macro": 0.6686,
    "f1_weighted": 0.6711,
    "precision_weighted": 0.6779,
    "recall_weighted": 0.6698,
    "confusion_matrix": [
      [
        269,
        145
      ],
      [
        98,
        224
      ]
    ],
    "num_samples": 736,
    "auc": 0.7343
  },
  {
    "model": "phobert-base",
    "scenario": "s2",
    "split": "dev",
    "accuracy": 0.717,
    "f1_macro": 0.7148,
    "f1_weighted": 0.7179,
    "precision_weighted": 0.7206,
    "recall_weighted": 0.717,
    "confusion_matrix": [
    

In [18]:
!cat /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/experiment_setup/runs/roberta_base_mm/roberta-base/summary.json

[
  {
    "model": "roberta-base",
    "scenario": "s1",
    "split": "dev",
    "accuracy": 0.7075,
    "f1_macro": 0.7013,
    "f1_weighted": 0.7066,
    "precision_weighted": 0.7063,
    "recall_weighted": 0.7075,
    "confusion_matrix": [
      [
        313,
        100
      ],
      [
        115,
        207
      ]
    ],
    "num_samples": 735,
    "auc": 0.7458
  },
  {
    "model": "roberta-base",
    "scenario": "s1",
    "split": "test",
    "accuracy": 0.6685,
    "f1_macro": 0.6627,
    "f1_weighted": 0.6682,
    "precision_weighted": 0.668,
    "recall_weighted": 0.6685,
    "confusion_matrix": [
      [
        294,
        120
      ],
      [
        124,
        198
      ]
    ],
    "num_samples": 736,
    "auc": 0.7125
  },
  {
    "model": "roberta-base",
    "scenario": "s2",
    "split": "dev",
    "accuracy": 0.6939,
    "f1_macro": 0.6918,
    "f1_weighted": 0.6949,
    "precision_weighted": 0.6985,
    "recall_weighted": 0.6939,
    "confusion_matrix": [
 

In [6]:
yaml_content = """extends: ../base.yaml

experiment:
  name: xlm_roberta_base_mm

run:
  scenarios: [s1, s2, s3, s4]

model:
  key: xlm-roberta-base
  family: text_classifier
  pretrained_name: xlm-roberta-base
"""

import os
config_path = "/kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/experiment_setup/configs/models/xlm_roberta_base.yaml"
os.makedirs(os.path.dirname(config_path), exist_ok=True)

with open(config_path, "w") as f:
    f.write(yaml_content)

# Verify
with open(config_path) as f:
    print(f.read())
print("✅ Done!")

extends: ../base.yaml

experiment:
  name: xlm_roberta_base_mm

run:
  scenarios: [s1, s2, s3, s4]

model:
  key: xlm-roberta-base
  family: text_classifier
  pretrained_name: xlm-roberta-base

✅ Done!


In [12]:
!python -m experiment_setup.main \
    --config experiment_setup/configs/models/xlm_roberta_base.yaml \
    --stage all \
    --scenario s1 

[run] output dir: /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/experiment_setup/runs/xlm_roberta_base_mm
[cache] preparing train split...
2026-06-02 01:59:32,170 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-02 01:59:32,171 - datasets - INFO - JAX version 0.7.2 available.
2026-06-02 01:59:33,357 - visonorm.normalizer - INFO - Loading tokenizer and model from visolex/visobert-normalizer-mix100...
2026-06-02 01:59:33,529 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/visolex/visobert-normalizer-mix100/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-02 01:59:33,530 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-02 01:59:33,548 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/visolex/visobert-normalizer-mix100/86731c2aa82948bd3bcbb682759d62c9c247588

In [13]:
!python -m experiment_setup.main \
    --config experiment_setup/configs/models/xlm_roberta_base.yaml \
    --stage all \
    --scenario s2

[run] output dir: /kaggle/working/vi-multimodal-sacarsm-detection-on-social-media/experiment_setup/runs/xlm_roberta_base_mm
[cache] preparing train split...
2026-06-02 02:24:14,084 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-02 02:24:14,085 - datasets - INFO - JAX version 0.7.2 available.
2026-06-02 02:24:15,323 - visonorm.normalizer - INFO - Loading tokenizer and model from visolex/visobert-normalizer-mix100...
2026-06-02 02:24:15,509 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/visolex/visobert-normalizer-mix100/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-02 02:24:15,509 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-02 02:24:15,527 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/visolex/visobert-normalizer-mix100/86731c2aa82948bd3bcbb682759d62c9c247588

In [ ]:
!python -m experiment_setup.main \
    --config experiment_setup/configs/models/xlm_roberta_base.yaml \
    --stage all \
    --scenario s3

In [ ]:
!python -m experiment_setup.main \
    --config experiment_setup/configs/models/xlm_roberta_base.yaml \
    --stage all \
    --scenario s4